In [ ]:
%run ./phase2_common

In [ ]:
STATE = f"{SCHEMA}.silver_child_snapshot_state"

dbutils.widgets.text("batch_id", "", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

bronze = spark.table(BRONZE).filter(F.col("batch_id") == batch_id)
if not bronze.limit(1).count():
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")
load_type = bronze.select("load_type").first()[0]

parsed = (
    bronze.select(
        "movie_id", "batch_id", "source_file", "source_extracted_at",
        F.from_json("raw_payload", MOVIE_SCHEMA).alias("movie"),
    )
    .select("movie_id", "batch_id", "source_file", "source_extracted_at", "movie.*")
)
if parsed.filter(F.col("id").isNull() | (F.col("id") != F.col("movie_id"))).limit(1).count():
    raise ValueError("Some Bronze records have a missing or mismatched movie ID")

COLLECTIONS = {
    "genres": dict(
        table="silver_movie_genres", array="genres", key="genre_id",
        columns=[("id", "genre_id", "BIGINT"), ("name", "genre_name", "STRING")],
        required=["genre_id", "genre_name"],
    ),
    "studios": dict(
        table="silver_movie_studios", array="production_companies", key="company_id",
        columns=[("id", "company_id", "BIGINT"), ("name", "company_name", "STRING"),
                 ("origin_country", "origin_country", "STRING")],
        required=["company_id", "company_name"],
    ),
    "cast": dict(
        table="silver_movie_cast", array="credits.cast", key="credit_id",
        columns=[("credit_id", "credit_id", "STRING"), ("id", "person_id", "BIGINT"),
                 ("name", "cast_name", "STRING"), ("character", "character", "STRING"),
                 ("order", "cast_order", "INT")],
        required=["credit_id"],
    ),
    "keywords": dict(
        table="silver_movie_keywords", array="keywords.keywords", key="keyword_id",
        columns=[("id", "keyword_id", "BIGINT"), ("name", "keyword_name", "STRING")],
        required=["keyword_id", "keyword_name"],
    ),
}

LINEAGE = (
    "source_batch_id STRING, source_file STRING, source_extracted_at TIMESTAMP, "
    "row_hash STRING, load_timestamp TIMESTAMP"
)
for cfg in COLLECTIONS.values():
    columns = ", ".join(f"{dst} {kind}" for _, dst, kind in cfg["columns"])
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SCHEMA}.{cfg['table']} (movie_id BIGINT, {columns}, {LINEAGE}) USING DELTA")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {STATE} (
    movie_id BIGINT, collection_name STRING, source_extracted_at TIMESTAMP,
    source_batch_id STRING, load_timestamp TIMESTAMP
) USING DELTA
""")

In [ ]:
def process(name, cfg):
    table, key = f"{SCHEMA}.{cfg['table']}", cfg["key"]
    present = parsed.filter(F.col(cfg["array"]).isNotNull())

    snapshots = present.select(
        "movie_id", "source_extracted_at", F.col("batch_id").alias("source_batch_id")
    )
    rows = (
        present.select(
            "movie_id", "batch_id", "source_file", "source_extracted_at",
            F.explode(cfg["array"]).alias("item"),
        )
        .select(
            "movie_id",
            *[F.col(f"item.{src}").alias(dst) for src, dst, _ in cfg["columns"]],
            F.col("batch_id").alias("source_batch_id"), "source_file", "source_extracted_at",
        )
    )

    if rows.filter(" OR ".join(f"`{c}` IS NULL" for c in cfg["required"])).limit(1).count():
        raise ValueError(f"{name}: rows are missing a required field")

    attributes = F.to_json(F.struct("movie_id", *[dst for _, dst, _ in cfg["columns"]]))
    conflicts = (
        rows.groupBy("movie_id", key).agg(F.countDistinct(attributes).alias("versions"))
        .filter("versions > 1")
    )
    if conflicts.limit(1).count():
        raise ValueError(f"{name}: conflicting attributes for the same key")

    rows = (
        rows.dropDuplicates(["movie_id", key])
        .withColumn("row_hash", F.sha2(attributes, 256))
        .withColumn("load_timestamp", F.current_timestamp())
    )

    stored = (
        spark.table(STATE).filter(F.col("collection_name") == name)
        .select("movie_id", F.col("source_extracted_at").alias("old_ts"), F.col("source_batch_id").alias("old_batch"))
    )
    newer = F.col("source_extracted_at") > F.col("old_ts")
    same_batch = (F.col("source_extracted_at") == F.col("old_ts")) & (F.col("source_batch_id") == F.col("old_batch"))
    eligible = snapshots.join(stored, "movie_id", "left").filter(F.col("old_batch").isNull() | newer | same_batch)
    if eligible.count() != snapshots.count():
        raise ValueError(f"{name}: some snapshots are older than stored data or tied; resolve ordering first")

    movie_ids = [r[0] for r in eligible.select("movie_id").collect()]
    if not movie_ids:
        return {"processed": 0}

    counts = upsert(table, rows, ["movie_id", key], delete_where=f"t.movie_id IN ({sql_list(movie_ids)})")
    upsert(
        STATE,
        eligible.select(
            "movie_id", F.lit(name).alias("collection_name"), "source_extracted_at",
            "source_batch_id", F.current_timestamp().alias("load_timestamp"),
        ),
        ["movie_id", "collection_name"],
    )
    return counts


for name, cfg in COLLECTIONS.items():
    step = f"batch_id={batch_id}; collection={name}"
    print(name, run_logged("Bronze-to-Silver", load_type, step, batch_id, lambda: process(name, cfg)))